In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

Задание №1. Загрузка данных
Изучить представленный набор данных на основе описания его столбцов в файле “horse_data.names” , загрузить его и оставить 8 столбцов для дальнейшего изучения: surgery?, Age, rectal temperature, pulse, respiratory rate, temperature of extremities, pain, outcome.

In [2]:
hd = pd.read_csv('horse_data.csv', 
                         encoding='utf-8', 
                         usecols=[0, 1, 3, 4,  5, 6, 10, 22], 
                         names=['surgery?', 'Age', 'rectal temperature', 'pulse', 'respiratory rate', 'temperature of extremities', 'pain', 'outcome'],
                         na_values=['?', ''])
hd.head()

,surgery?,Age,rectal temperature,pulse,respiratory rate,temperature of extremities,pain,outcome
0,2.0,1,38.5,66.0,28.0,3.0,5.0,2.0
1,1.0,1,39.2,88.0,20.0,NaN,3.0,3.0
2,2.0,1,38.3,40.0,24.0,1.0,3.0,1.0
3,1.0,9,39.1,164.0,84.0,4.0,2.0,2.0
4,2.0,1,37.3,104.0,35.0,NaN,NaN,2.0


Задание №2. Первичное изучение данных
Проанализировать значения по столбцам, рассчитать базовые статистики, найти выбросы.

In [13]:
numeric_column = ["rectal temperature", "pulse", "respiratory rate"]
category_column = ["surgery?", "Age", "temperature of extremities", "pain", "outcome"]


In [ ]:
print("Числовые базовые статистики:")
for column in numeric_column:
    print(f"{column} mean: {hd[column].mean()}") #Арифметическое среднее
    print(f"{column} mode: {hd[column].round().mode()[0]}") #Мода
    print(f"{column} median: {hd[column].median()}") #Медиана
    print(f"{column} std: {hd[column].std()}") #СКО
    print(f"{column} var: {hd[column].var()}") #Дисперсия
    print(f"{column} na count {hd[column].isna().sum()} out of {len(hd[column])}\n") #Кол-во na

In [ ]:
print(f"\nКатегориальные базовые статистики:")
for column in category_column:
    print(f"{column} value counts: {hd[column].value_counts()}") #Число вхождений
    print(f"{column} count unique: {hd[column].nunique()}") #Количество уникальных значений
    print(f"{column} unique value: {hd[column].unique()}") #Уникальные значения
    print(f"{column} mode: {hd[column].mode()[0]}\n") #Мода
    print(f"{column} na count {hd[column].isna().sum()} out of {len(hd[column])}\n") #Кол-во na

In [ ]:
def detect_outliers_iqr(column):
    q1 = hd[column].quantile(0.25)
    q3 = hd[column].quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - (1.5 * iqr)
    upper_bound = q3 + (1.5 * iqr)
    outliers =  hd[~hd[column].between(lower_bound, upper_bound, inclusive = "both")]
    return outliers[column].dropna()

print("Выбросы в числовых колонках: ")
for column in numeric_column:
    print(detect_outliers_iqr(column))
    


In [ ]:
print("Выбросы в числовых колонках: ")
for column in numeric_column:
    print(detect_outliers_iqr(column))

In [ ]:
print("Выбросы в категориальных колонках: ")
print(hd[~hd["surgery?"].isin([1, 2])]["surgery?"].dropna())
print(hd[~hd["Age"].isin([1, 2])]["Age"].dropna())
print(hd[~hd["temperature of extremities"].isin([1, 2, 3, 4])]["temperature of extremities"].dropna())
print(hd[~hd["pain"].isin([1, 2, 3, 4, 5])]["pain"].dropna())
print(hd[~hd["outcome"].isin([1, 2, 3])]["outcome"].dropna())


Задание 3. Работа с пропусками
Рассчитать количество пропусков для всех выбранных столбцов. Принять и обосновать решение о методе заполнения пропусков по каждому столбцу на основе рассчитанных статистик и возможной взаимосвязи значений в них. Сформировать датафрейм, в котором пропуски будут отсутствовать.

In [ ]:
((hd.isna().mean()) * 100).round(2)

surgery?                       0.33
Age                            0.00
rectal temperature            20.00
pulse                          8.00
respiratory rate              19.33
temperature of extremities    18.67
pain                          18.33
outcome                        0.33
dtype: float64

In [ ]:
hd = hd.dropna(subset=['outcome']) #Отсутствие outcome не позволит нам использовать данные для обучения модели
hd[numeric_column] = hd[numeric_column].fillna(hd[numeric_column].median()) #Числовые колонки заполняем медианой, так как она не чувствительна к выбросам
hd[category_column] = hd[category_column].fillna("Unknown") #Категориальные колонки заполняем "Unknown", так как это позволит нам сохранить информацию о том, что данные отсутствуют, и не исказит распределение данных
((hd.isna().mean()) * 100).round(2)

surgery?                      0.0
Age                           0.0
rectal temperature            0.0
pulse                         0.0
respiratory rate              0.0
temperature of extremities    0.0
pain                          0.0
outcome                       0.0
dtype: float64